# Project File Sharing: Audit and Repair

Files copied into a **My Projects** area from the command line (scp, cp, mv) often end up invisible or read-only for other project members, even though project membership is fine. The cause is POSIX: project sharing works through per-member ACL entries plus an ACL *mask*, and command-line tools either cap the mask with a restrictive file mode or wipe the member entries entirely.

This notebook, run from anywhere with dapi:

1. installs the dapi `dev` build,
2. **audits** every file in the project (who can actually see what),
3. breaks a demo file on purpose and repairs it with `fix_permissions()`,
4. re-audits to confirm.

`fix_permissions` picks the strongest repair each file allows: direct ACL repair for Tapis-owned files, an **owner tier** through the `cloud.data` system (which acts as *you* on the storage host, so your own transfers are fixable from any machine), copy-recreate for files the service account can read, and for files owned by another member it reports the exact call for that person to run.

In [ ]:
%pip install --quiet --upgrade "dapi @ git+https://github.com/DesignSafe-CI/dapi.git@dev"

**Restart the kernel once after the install** (Kernel → Restart) so the new dapi is imported, then run from the next cell.

In [ ]:
import dapi
from dapi import DSClient

assert hasattr(dapi.projects, "fix_project_permissions"), (
    "Old dapi is still loaded. Restart the kernel and rerun from here."
)
ds = DSClient()

PROJECT = "PRJ-6457"
EXCLUDE = {"scp600.txt"}  # preserved evidence, do not repair

## 1. Audit: who can actually see each file

For every member: their Tapis grant, their named POSIX ACL entry, the mask that caps it, the file's world bits, and the resulting **effective** access. Healthy Tapis-written files show `rw-`; command-line arrivals show the two diseases (`mask` capped, or entry `missing`).

In [ ]:
def audit(project):
    for f in ds.projects.files(project, output="raw"):
        print(f"\n=== {f.name}  (mode: {f.nativePermissions}) ===")
        print(
            ds.projects.permissions(project, "/" + f.name, output="df").to_string(
                index=False
            )
        )


audit(PROJECT)

## 2. Break a file on purpose

Writing through the JupyterHub mount creates the file as *you* and inherits the project's member ACLs; a `chmod 600` then caps the mask, exactly what `scp` of a private file does. This is the reproducible demo case.

In [ ]:
# Create a file owned by YOU (via cloud.data, which acts as you on the
# storage host), then break it the way scp does: cap the ACL mask.
import os
import tempfile

ROOT = "corral-repl/projects/NHERI/projects/e78a78c4-457d-4d81-99ed-b9d8dee56a11"
demo = "demo-broken.txt"

tmp = os.path.join(tempfile.mkdtemp(), demo)
with open(tmp, "w") as f:
    f.write("demo: broken on purpose\n")
ds.files.upload(tmp, f"tapis://cloud.data/{ROOT}/{demo}")

ds.tapis.files.setFacl(
    systemId="cloud.data",
    path=f"{ROOT}/{demo}",
    operation="ADD",
    recursionMethod="NONE",
    aclString="mask::---",
)

print(ds.projects.permissions(PROJECT, "/" + demo, output="df").to_string(index=False))

Members show `effective: none`. Membership is intact; the mask vetoes it.

## 3. Preview the repair (nothing changes)

`dry_run=True` classifies every file and reports what the fix *would* do, including which local project mount it auto-detected.

In [ ]:
import json

plan = ds.projects.fix_permissions(PROJECT, dry_run=True)
print(json.dumps(plan, indent=1))

## 4. Repair, leaving the evidence file alone

One call per file here so the output shows each file's strategy. In practice, `ds.projects.fix_permissions(PROJECT)` handles the whole project at once.

In [ ]:
for f in ds.projects.files(PROJECT, output="raw"):
    if f.name in EXCLUDE:
        print(f"{f.name}: skipped (preserved evidence)")
        continue
    r = ds.projects.fix_permissions(PROJECT, "/" + f.name)
    if r["fixed"]:
        print(f"{f.name}: {list(r['fixed'].values())[0]}")
    elif r["unfixable"]:
        print(f"{f.name}: UNFIXABLE here -> {r['unfixable'][0]['owner_fix']}")

## 5. Re-audit

In [ ]:
audit(PROJECT)

## The complete repair story

- **Your own broken transfers**: `ds.projects.fix_permissions(PROJECT, "/file")` fixes them from anywhere via the owner tier (`cloud.data` acts as you on the storage host).
- **Tapis-owned files** (including files predating a newly added member): fixed directly.
- **Another member's broken transfers**: the report hands you the exact `fix_permissions` call for *them* to run.

**Prevention beats repair:** transfer into projects through Tapis (portal, dapi, or job archiving into the project system), or, when using `cp`/`scp` from a login node, finish with `chmod -R g+rwX` on the destination. Never `mv`, `cp -p`, or `rsync -a` into a project.